# Physics-Informed Neural Network (PINN) for Aerofoil Flow

Predicts **velocity (u, v)** and **pressure (p)** fields around an aerofoil
by embedding Navier-Stokes equations into the neural network loss function.

**Dataset:** 3,668 points from ANSYS Fluent CFD simulation of an aerofoil
**Framework:** PyTorch | **Physics:** 2D Incompressible Navier-Stokes

In [ ]:
import pandas as pd
import torch
import torch.nn as nn
import matplotlib.pyplot as plt
from torch.optim.lr_scheduler import ReduceLROnPlateau
import os

# ── Configuration ──────────────────────────────────────────────────────────
DATA_FILE = 'pinn_aerofoil_data.csv'   # dataset file (same folder as notebook)
EPOCHS    = 20000
LR        = 1e-2
HIDDEN    = 200
VISCOSITY = 0.01                        # kinematic viscosity (m²/s)

print(f'PyTorch version : {torch.__version__}')
print(f'Dataset file    : {DATA_FILE}')
print(f'File exists     : {os.path.exists(DATA_FILE)}')


## 1. Load & Preprocess Data

In [ ]:
# Column names matching ANSYS export format
column_names = ['X [m]', 'Y [m]', 'Pressure [Pa]',
                'Velocity u.Gradient [s^-1]', 'Velocity v.Gradient [s^-1]']

# Load CSV — skip 4 metadata rows, assign column names
data = pd.read_csv(DATA_FILE, skiprows=4, names=column_names, header=None)

# Convert to numeric, drop NaN and duplicates
for col in column_names:
    data[col] = pd.to_numeric(data[col], errors='coerce')
data = data.dropna().drop_duplicates().reset_index(drop=True)

print(f'Loaded {len(data)} data points')
print(data.describe().round(4))


In [ ]:
# Convert to tensors
x  = torch.tensor(data['X [m]'].values.reshape(-1, 1),                    dtype=torch.float32)
y  = torch.tensor(data['Y [m]'].values.reshape(-1, 1),                    dtype=torch.float32)
p  = torch.tensor(data['Pressure [Pa]'].values.reshape(-1, 1),            dtype=torch.float32)
vx = torch.tensor(data['Velocity u.Gradient [s^-1]'].values.reshape(-1,1),dtype=torch.float32)
vy = torch.tensor(data['Velocity v.Gradient [s^-1]'].values.reshape(-1,1),dtype=torch.float32)

# Normalization statistics (mean / std)
def norm_stats(t):
    std = t.std()
    return t.mean(), (std if std != 0 else torch.tensor(1.0))

x_mean,  x_std  = norm_stats(x)
y_mean,  y_std  = norm_stats(y)
vx_mean, vx_std = norm_stats(vx)
vy_mean, vy_std = norm_stats(vy)
p_mean,  p_std  = norm_stats(p)

# Normalised tensors
x_norm  = (x  - x_mean)  / x_std
y_norm  = (y  - y_mean)  / y_std
vx_norm = (vx - vx_mean) / vx_std
vy_norm = (vy - vy_mean) / vy_std
p_norm  = (p  - p_mean)  / p_std
xy_norm = torch.cat([x_norm, y_norm], dim=1)

print('Normalisation complete.')
print(f'  x  : mean={x_mean:.4f}, std={x_std:.4f}')
print(f'  y  : mean={y_mean:.4f}, std={y_std:.4f}')
print(f'  vx : mean={vx_mean:.4f}, std={vx_std:.4f}')
print(f'  p  : mean={p_mean:.4f},  std={p_std:.4f}')


## 2. PINN Architecture

Three **separate networks** — one each for u, v, p — sharing the same architecture:
```
Input(2) → Linear(200) → Tanh → Linear(200) → Tanh → Linear(1)
```

In [ ]:
class PINN(nn.Module):
    """Fully-connected network: 2 hidden layers, Tanh activations."""
    def __init__(self, input_dim=2, hidden_dim=HIDDEN):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(input_dim, hidden_dim), nn.Tanh(),
            nn.Linear(hidden_dim, hidden_dim), nn.Tanh(),
            nn.Linear(hidden_dim, 1)
        )
        # Xavier-style initialisation
        for layer in self.net:
            if isinstance(layer, nn.Linear):
                layer.weight.data.normal_(mean=0, std=0.1)
                layer.bias.data.fill_(0)

    def forward(self, coords):
        return self.net(coords)

# One model per output variable
model_u = PINN()
model_v = PINN()
model_p = PINN()

total_params = sum(p.numel() for p in model_u.parameters()) * 3
print(f'Total trainable parameters (3 models): {total_params:,}')


## 3. Loss Functions

**Total Loss = 10 × Data Loss + 0.001 × Physics Loss + 10 × Boundary Loss**

- **Data loss** — MSE between predictions and ANSYS ground truth  
- **Physics loss** — residuals of continuity + x/y momentum (Navier-Stokes)  
- **Boundary loss** — no-slip condition at domain corners

In [ ]:
# Boundary condition points (4 corners of domain)
x_min, x_max = x.min(), x.max()
y_min, y_max = y.min(), y.max()

bc_x      = torch.tensor([[x_min, y_min],[x_max, y_min],
                           [x_min, y_max],[x_max, y_max]], dtype=torch.float32)
bc_x_norm = (bc_x - torch.tensor([[x_mean, y_mean]])) / torch.tensor([[x_std, y_std]])
bc_u      = torch.zeros((4, 1))                 # no-slip: u=0
bc_v      = torch.zeros((4, 1))                 # no-slip: v=0
bc_p      = torch.mean(p) * torch.ones((4, 1))  # mean pressure at boundary


def compute_residuals(model_u, model_v, model_p, xy_n):
    """Compute Navier-Stokes PDE residuals via automatic differentiation."""
    xy_n = xy_n.clone().requires_grad_(True)

    # Denormalise predictions
    u = model_u(xy_n) * vx_std + vx_mean
    v = model_v(xy_n) * vy_std + vy_mean
    p_pred = model_p(xy_n) * p_std + p_mean

    def grad(out, wrt): return torch.autograd.grad(out.sum(), wrt, create_graph=True)[0]

    gu = grad(u, xy_n);      u_x = gu[:,0:1]/x_std;  u_y = gu[:,1:2]/y_std
    gv = grad(v, xy_n);      v_x = gv[:,0:1]/x_std;  v_y = gv[:,1:2]/y_std
    gp = grad(p_pred, xy_n); p_x = gp[:,0:1]/x_std;  p_y = gp[:,1:2]/y_std

    u_xx = grad(u_x, xy_n)[:,0:1]/x_std
    u_yy = grad(u_y, xy_n)[:,1:2]/y_std
    v_xx = grad(v_x, xy_n)[:,0:1]/x_std
    v_yy = grad(v_y, xy_n)[:,1:2]/y_std

    continuity = (u_x + v_y)                              / (vx_std / x_std)
    momentum_u = (u*u_x + v*u_y + p_x - VISCOSITY*(u_xx+u_yy)) / (vx_std / x_std)
    momentum_v = (u*v_x + v*v_y + p_y - VISCOSITY*(v_xx+v_yy)) / (vy_std / y_std)

    return continuity, momentum_u, momentum_v


def total_loss(xy_n, vx_n, vy_n, p_n):
    mse = nn.MSELoss()

    # 1. Data loss
    u_pred = model_u(xy_n) * vx_std + vx_mean
    v_pred = model_v(xy_n) * vy_std + vy_mean
    p_pred_d = model_p(xy_n) * p_std + p_mean
    data_loss = mse(u_pred, vx) + mse(v_pred, vy) + mse(p_pred_d, p)

    # 2. Physics loss (Navier-Stokes residuals)
    cont, mom_u, mom_v = compute_residuals(model_u, model_v, model_p, xy_n)
    physics_loss = (cont**2).mean() + (mom_u**2).mean() + (mom_v**2).mean()

    # 3. Boundary loss
    bc_loss = (mse(model_u(bc_x_norm)*vx_std+vx_mean, bc_u) +
               mse(model_v(bc_x_norm)*vy_std+vy_mean, bc_v) +
               mse(model_p(bc_x_norm)*p_std +p_mean,  bc_p))

    return 10.0*data_loss + 0.001*physics_loss + 10.0*bc_loss

print('Loss functions defined.')


## 4. Training

In [ ]:
optimizer_u = torch.optim.Adam(model_u.parameters(), lr=LR)
optimizer_v = torch.optim.Adam(model_v.parameters(), lr=LR)
optimizer_p = torch.optim.Adam(model_p.parameters(), lr=LR)
scheduler_u = ReduceLROnPlateau(optimizer_u, mode='min', factor=0.5, patience=50)
scheduler_v = ReduceLROnPlateau(optimizer_v, mode='min', factor=0.5, patience=50)
scheduler_p = ReduceLROnPlateau(optimizer_p, mode='min', factor=0.5, patience=50)

loss_history = []
best_loss    = float('inf')

for epoch in range(EPOCHS):
    model_u.train(); model_v.train(); model_p.train()
    optimizer_u.zero_grad(); optimizer_v.zero_grad(); optimizer_p.zero_grad()

    loss = total_loss(xy_norm, vx_norm, vy_norm, p_norm)
    loss.backward()
    optimizer_u.step(); optimizer_v.step(); optimizer_p.step()
    scheduler_u.step(loss); scheduler_v.step(loss); scheduler_p.step(loss)

    loss_val = loss.item()
    loss_history.append(loss_val)

    if epoch % 500 == 0:
        print(f'Epoch {epoch:>6} | Loss: {loss_val:.5e}')

    if loss_val < best_loss:
        best_loss = loss_val
        torch.save({'u': model_u.state_dict(),
                    'v': model_v.state_dict(),
                    'p': model_p.state_dict()}, 'best_models.pth')

print(f'\nTraining complete. Best loss: {best_loss:.5e}')


## 5. Results & Visualisation

In [ ]:
# Load best checkpoint and evaluate
checkpoint = torch.load('best_models.pth')
model_u.load_state_dict(checkpoint['u'])
model_v.load_state_dict(checkpoint['v'])
model_p.load_state_dict(checkpoint['p'])

model_u.eval(); model_v.eval(); model_p.eval()
with torch.no_grad():
    u_pred = model_u(xy_norm) * vx_std + vx_mean
    v_pred = model_v(xy_norm) * vy_std + vy_mean
    p_pred = model_p(xy_norm) * p_std  + p_mean

# ── L2 Relative Error (key metric for papers/resume) ──────────────────────
def l2_error(pred, true):
    return (torch.norm(pred - true) / torch.norm(true)).item()

print('── L2 Relative Errors ──────────────────')
print(f'  Velocity u : {l2_error(u_pred, vx)*100:.2f}%')
print(f'  Velocity v : {l2_error(v_pred, vy)*100:.2f}%')
print(f'  Pressure p : {l2_error(p_pred, p)*100:.2f}%')

u_error = torch.abs(u_pred - vx)
v_error = torch.abs(v_pred - vy)
p_error = torch.abs(p_pred - p)

# ── Training loss curve ────────────────────────────────────────────────────
plt.figure(figsize=(8,3))
plt.semilogy(loss_history)
plt.title('Training Loss (log scale)')
plt.xlabel('Epoch'); plt.ylabel('Loss')
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig('loss_curve.png', dpi=150)
plt.show()

# ── ANSYS vs PINN comparison ───────────────────────────────────────────────
fig, axes = plt.subplots(1, 3, figsize=(16, 5))
titles = ['Velocity u [s⁻¹]', 'Velocity v [s⁻¹]', 'Pressure [Pa]']
ansys  = [vx, vy, p]
pinn   = [u_pred, v_pred, p_pred]

for ax, title, a, pr in zip(axes, titles, ansys, pinn):
    ax.scatter(x.numpy(), a.numpy(),  c='steelblue',  alpha=0.4, s=8, label='ANSYS')
    ax.scatter(x.numpy(), pr.numpy(), c='darkorange', alpha=0.4, s=8, label='PINN')
    ax.set_title(title); ax.set_xlabel('X [m]'); ax.legend()

plt.suptitle('ANSYS vs PINN Predictions', fontsize=13, y=1.02)
plt.tight_layout()
plt.savefig('pinn_comparison.png', dpi=150, bbox_inches='tight')
plt.show()

# ── Absolute error plots ───────────────────────────────────────────────────
fig, axes = plt.subplots(1, 3, figsize=(16, 5))
errors = [u_error, v_error, p_error]
ylabels = ['Error u [s⁻¹]', 'Error v [s⁻¹]', 'Error p [Pa]']

for ax, err, ylabel in zip(axes, errors, ylabels):
    sc = ax.scatter(x.numpy(), err.numpy(), c=err.numpy(), cmap='viridis', s=8, alpha=0.6)
    plt.colorbar(sc, ax=ax, label='Abs Error')
    ax.set_title(f'Absolute Error — {ylabel.split("[")[0].strip()}')
    ax.set_xlabel('X [m]'); ax.set_ylabel(ylabel)

plt.suptitle('Pointwise Absolute Errors', fontsize=13, y=1.02)
plt.tight_layout()
plt.savefig('pinn_errors.png', dpi=150, bbox_inches='tight')
plt.show()

# ── 2D Flow field (X-Y scatter coloured by pressure) ──────────────────────
plt.figure(figsize=(10, 4))
sc = plt.scatter(x.numpy(), y.numpy(), c=p_pred.numpy(), cmap='coolwarm', s=5, alpha=0.7)
plt.colorbar(sc, label='Predicted Pressure [Pa]')
plt.title('PINN Predicted Pressure Field — Aerofoil')
plt.xlabel('X [m]'); plt.ylabel('Y [m]')
plt.tight_layout()
plt.savefig('pressure_field.png', dpi=150)
plt.show()

print('All plots saved.')
